In [75]:
import re
import numpy as np
from math import log

## Build dataset

In [3]:
train_data = [
    ("I love this movie", 1),
    ("This film is amazing", 1),
    ("I hate this movie", 0),
    ("This film is terrible", 0)
]


comvention:
- 1 → positve
- 0 → negative

In [4]:
for sentence, label in train_data:
    print(f'Sentence: {sentence}')
    print(f'label: {label}')
    print()

Sentence: I love this movie
label: 1

Sentence: This film is amazing
label: 1

Sentence: I hate this movie
label: 0

Sentence: This film is terrible
label: 0



In [5]:
positive_count = 0
negative_count = 0

total_sentence = len (train_data)

for sentence, label in train_data:
    if label == 1:
        positive_count +=1
    else:
        negative_count +=1

p_positive = positive_count/total_sentence
n_positive = negative_count/total_sentence

print("positive prior", p_positive)
print("negative prior", n_positive)

positive prior 0.5
negative prior 0.5


Current model only knows:
- how common positive reviews are
- how common negative reviews are 

## Preprocess text
The preprocessing pipeline usually includes:
- lowercase conversion
- punctuation removal
- tokenization
- optional stop word removal
- optional stemming

In [6]:
def preprocess (text):
    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = text.split()
    
    return tokens

In [8]:
train_data = [
    ("I love this movie", 1),
    ("This film is amazing", 1),
    ("What a great experience", 1),
    ("I really enjoyed this", 1),

    ("I hate this movie", 0),
    ("This film is terrible", 0),
    ("What a bad experience", 0),
    ("I really dislike this", 0)
]

In [9]:
train_data[1]

('This film is amazing', 1)

In [ ]:
def build_freqs(train_data):
    freqs = {}

    for tub in train_data:
        
        sentence = tub[0]        
        label = tub[1]
        
        tokens = preprocess (sentence)
        for token in tokens:
            pair = (token,label) # set  -> membership / uniqueness

            if pair in freqs:
                freqs[(pair)] += 1 # dict -> mapping from thing to value
            
            else:
                freqs[(pair)] = 1
            

    return(freqs)



In [15]:
freqs = {
    ("love", 1): 1,
    ("movie", 1): 1,
    ("movie", 0): 1,
    ("terrible", 0): 1
}

In [25]:
def build_vocab(freqs):
    pairs = freqs.keys()
    vocab = set ()
    
    for pair in pairs:
        vocab.add(pair [0])
    
    return (vocab)

In [26]:
print(build_vocab(freqs))

{'terrible', 'love', 'movie'}


## Convert text into statistical information model can use

$$P(word ∣ class) = \frac{\text{count ( word, class)} ​}{\text{total words in class}}$$

- After smoothing:
$$P(word ∣ class) = \frac{\text{count ( word, class)} ​ + 1​ }{\text{total words in class} + V}$$

where V is the vocabulary size.

The +1 prevents zero probability.


In [57]:
freqs = {
    ("love", 1): 2,
    ("movie", 1): 3,
    ("great", 1): 1,

    ("hate", 0): 2,
    ("movie", 0): 1,
    ("bad", 0): 3
}

n_pos = 6   # 2 + 3 + 1
n_neg = 6   # 2 + 1 + 3

vocab = {"love", "movie", "great", "hate", "bad"}
V = len(vocab)

In [30]:
word_count = freqs.get(("love", 1), 0)
word_count

2

In [ ]:
def get_word_probability(freqs, word, label, n_pos, n_neg, V):
    
    word_count = freqs.get((word, label), 0)
    if label == 1:
        denominator = n_pos + V
        
    else:
        denominator = n_neg + V
        
    prob = (word_count + 1) / (denominator)

    return prob

In [ ]:
def count_words_by_label(freqs):

    n_pos = 0
    n_neg = 0


    for (word, label), count in freqs.items(): 
        # we need total word occurrences, so we add the frequency value itself.
        if label == 1:
            n_pos += count 

        else:
            n_neg += count

    return n_pos, n_neg

In [72]:
print(count_words_by_label(freqs))

(6, 6)


**The probability of sentence**

$$ P(sentence∣positive) = P(w1​∣positive)×P(w2​∣positive)×...×P(wn​∣positive)  $$

Multiplying many tiny probabilities can become extremely small. So in real implementations, we usually use log and add instead of multiply.

The solution is the logarithm identity:
$$ log(a×b)=log(a)+log(b) $$

We compute:
$$ log(P(positive))+log(P(w1​∣positive))+log(P(w2​∣positive))+... $$


In [ ]:
def score_sentence(sentence, label, freqs, vocab, n_pos, n_neg, p_pos, p_neg):

    if label == 1:
        score = log(p_pos)
    else:
        score = log(p_neg) 

    
    tokens = preprocess(sentence)
    
    for token in tokens:
        if token in vocab: # means: ignore words the model never saw during training.
            prob = get_word_probability(freqs, token, label, n_pos, n_neg, len(vocab))
            score += log(prob)
            

    return score
    
    